<a href="https://colab.research.google.com/github/ugisrutinsRSU/RSU_Colab/blob/main/03_Simple_Neural_Network_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple Neural Network Classifier in PyTorch

## Overview

In this exercise you will build, train, and evaluate a **binary classification neural network** using **PyTorch**.

The business context is **customer purchase intention**: given a user's browsing behaviour on an e-commerce site, predict whether they will complete a purchase during that session.

By the end of this notebook you will have practiced:
- Loading and preprocessing a real-world tabular dataset
- Encoding categorical features and normalising numerical ones
- Building a feedforward neural network with `torch.nn.Module`
- Writing a training loop with a loss function and optimiser
- Evaluating model performance with accuracy, precision, recall, and F1-score
- Visualising training progress


## Dataset: Online Shoppers Purchasing Intention

**Source:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Online+Shoppers+Purchasing+Intention+Dataset)

The dataset contains 12,330 sessions collected from an e-commerce website. Each row represents one user session and includes:

| Feature group | Examples |
|---|---|
| Page visit counts | `Administrative`, `Informational`, `ProductRelated` |
| Time spent on pages | `Administrative_Duration`, `Informational_Duration`, `ProductRelated_Duration` |
| Google Analytics metrics | `BounceRates`, `ExitRates`, `PageValues`, `SpecialDay` |
| Session context | `Month`, `OperatingSystems`, `Browser`, `Region`, `TrafficType` |
| Visitor type | `VisitorType` (Returning / New / Other) |
| Timing flag | `Weekend` (Boolean) |

**Target variable:** `Revenue` — `True` if the session ended in a purchase, `False` otherwise.

The dataset is **imbalanced**: roughly 84 % of sessions do not result in a purchase.

### Getting the data

Run the cell below to download the CSV directly from the UCI repository.

In [ ]:
import urllib.request
import os

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00468/online_shoppers_intention.csv"
filename = "online_shoppers_intention.csv"

if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)
    print("Dataset downloaded.")
else:
    print("Dataset already present.")

## Step 1 — Imports

Import all libraries you will need for this exercise.

**You will need at minimum:**
- `pandas` and `numpy` for data handling
- `sklearn` utilities: `train_test_split`, `StandardScaler`, and classification metrics
- `torch`, `torch.nn`, `torch.optim`
- `torch.utils.data`: `TensorDataset`, `DataLoader`
- `matplotlib.pyplot` for plotting

In [ ]:
# Your imports here


## Step 2 — Load and Explore the Data

Load the CSV file into a pandas DataFrame.

**Tasks:**
1. Load the dataset and display the first few rows.
2. Check the shape, column names, and data types.
3. Check for missing values.
4. Inspect the class balance: how many sessions resulted in a purchase vs. not?


In [ ]:
# Load the dataset


In [ ]:
# Explore shape, dtypes, missing values, and class balance


## Step 3 — Preprocess the Data

The dataset contains a mix of numerical and categorical columns. You need to prepare it before passing it to a neural network.

**Tasks:**
1. **Encode categorical columns.** The columns `Month`, `VisitorType`, and `Weekend` are not numeric. Use one-hot encoding (e.g., `pd.get_dummies`) or label encoding as appropriate. Drop the original columns after encoding.
2. **Encode the target.** Convert the `Revenue` column (boolean) to integer (0/1) and separate it from the features.
3. **Split the data** into training and test sets (80 % / 20 %). Use `random_state=42` and `stratify=y` to preserve class proportions.
4. **Normalise numerical features** using `StandardScaler`. Fit the scaler on the training set only, then transform both train and test sets.

> **Why stratify?** The dataset is imbalanced. Stratified splitting ensures both train and test sets have the same proportion of positive examples as the full dataset.

In [ ]:
# Encode categorical features and target


In [ ]:
# Train/test split and feature normalisation


## Step 4 — Create PyTorch Datasets and DataLoaders

PyTorch models consume data through `DataLoader` objects, which handle batching and shuffling automatically.

**Tasks:**
1. Convert your NumPy arrays `X_train`, `X_test`, `y_train`, `y_test` to `torch.FloatTensor` (features) and `torch.FloatTensor` (labels — keep as float for `BCELoss` compatibility).
2. Wrap each pair into a `TensorDataset`.
3. Create a `DataLoader` for the training set with `batch_size=64` and `shuffle=True`, and one for the test set with `shuffle=False`.
4. Print the number of batches in each loader to verify.

In [ ]:
# Convert to tensors, create TensorDatasets and DataLoaders


## Step 5 — Define the Neural Network

Build a feedforward neural network by subclassing `torch.nn.Module`.

**Architecture requirements:**
- **Input layer:** size equal to the number of features after preprocessing.
- **Hidden layer 1:** 64 neurons, ReLU activation.
- **Hidden layer 2:** 32 neurons, ReLU activation.
- **Output layer:** 1 neuron, Sigmoid activation (outputs a probability between 0 and 1).

**Tasks:**
1. Define the class `CustomerClassifier(nn.Module)` with an `__init__` method that builds the layers and a `forward` method that defines the data flow.
2. Instantiate the model and print it to verify the architecture.

> **Tip:** `nn.Sequential` can help you chain layers cleanly inside `__init__`.

In [ ]:
# Define and instantiate the neural network


## Step 6 — Define Loss Function and Optimiser

For binary classification with a sigmoid output, the standard choice is **Binary Cross-Entropy loss**.

**Tasks:**
1. Instantiate `nn.BCELoss()` as your loss function.
2. Instantiate `torch.optim.Adam` with a learning rate of `0.001` as your optimiser, passing `model.parameters()`.

> **Why Adam?** Adam adapts the learning rate per parameter and generally converges faster than plain SGD on tabular data.

In [ ]:
# Define loss function and optimiser


## Step 7 — Write the Training Loop

Train the model for **30 epochs**. For each epoch you should:

1. Set the model to training mode with `model.train()`.
2. Iterate over batches from `train_loader`.
3. For each batch:
   - Zero the gradients with `optimizer.zero_grad()`.
   - Run a **forward pass** to get predictions.
   - Compute the **loss**.
   - Run a **backward pass** with `loss.backward()`.
   - Update the weights with `optimizer.step()`.
4. After all batches, record the average epoch training loss.
5. Run a **validation pass** (no gradient computation) over `test_loader` and record the average test loss.
6. Print the losses every 5 epochs.

Store training and test losses in lists so you can plot them in the next step.

In [ ]:
# Training loop


## Step 8 — Visualise Training Progress

Plot the training and test loss curves over epochs.

**Tasks:**
1. Create a line plot with epochs on the x-axis and loss on the y-axis.
2. Show both training loss and test loss on the same plot with a legend.
3. Add axis labels and a title.

> **What to look for:** If training loss decreases but test loss plateaus or rises, the model may be overfitting.

In [ ]:
# Plot training and test loss curves


## Step 9 — Evaluate the Model

A single accuracy number is not enough for an imbalanced dataset. Evaluate using a full set of metrics.

**Tasks:**
1. Set the model to evaluation mode with `model.eval()`.
2. Run inference over the entire test set (disable gradient tracking with `torch.no_grad()`).
3. Convert predicted probabilities to binary labels using a threshold of 0.5.
4. Compute and print:
   - **Accuracy**
   - **Precision**
   - **Recall**
   - **F1-score**
5. Print a **classification report** using `sklearn.metrics.classification_report`.

> **Tip:** Use `sklearn.metrics` functions. Remember to move tensors to CPU and convert to NumPy before passing to sklearn.

In [ ]:
# Evaluate the model on the test set
